In [1]:

from claimbuster_spotter.adv_transformer.core.utils.flags import FLAGS


In [2]:
import os
# display current working directory
os.getcwd()

'/home/adamj/factcheck-podcasts/src'

In [3]:
FLAGS.cs_model_dir = "/home/adamj/factcheck-podcasts/src/claimbuster_spotter/output/bb/"

In [4]:
from claimbuster_spotter.adv_transformer.core.api.api_wrapper import ClaimSpotterAPI
claimspotter = ClaimSpotterAPI()

2023-04-11 11:19:11.325800: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
[nltk_data] Downloading package punkt to /home/adamj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/adamj/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets to /home/adamj/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package stopwords to /home/adamj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dependencies Loaded.


2023-04-11 11:19:13.680943: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2023-04-11 11:19:13.695962: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:923] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-04-11 11:19:13.695994: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4090 computeCapability: 8.9
coreClock: 2.52GHz coreCount: 128 deviceMemorySize: 23.99GiB deviceMemoryBandwidth: 938.86GiB/s
2023-04-11 11:19:13.696009: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2023-04-11 11:19:13.697885: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcublas.so.11
2023-04-11 11:19:13.697958: I tensorflow/stream_execut

In [5]:
claimspotter.single_sentence_query("This is a test")

2023-04-11 11:19:21.737525: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)
2023-04-11 11:19:21.743221: I tensorflow/core/platform/profile_utils/cpu_utils.cc:114] CPU Frequency: 3187195000 Hz


array([[0.70457386, 0.29542614]])

In [6]:
sentence_list = [
    'Donald Trump is the 45th President of the United States',
    'I really like cheese',
    'McDonalds earns $10 billion dollars each minute'
]

In [7]:
claimspotter.batch_sentence_query(sentence_list)

array([[0.38129014, 0.61870986],
       [0.76754425, 0.23245575],
       [0.21547081, 0.78452919]])

In [8]:
import requests

In [9]:
podcasts = requests.get("http://127.0.0.1:8008/api/podcasts/")
podcasts = podcasts.json()

In [11]:
# get the uuid field of each segmentation object in each segmentation_set for each transcription in transcription_set and each audioitem in audioitem_set and each podcast in podcasts
segmentation_uuids = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                segmentation_uuids.append(segmentation['uuid'])
len(segmentation_uuids)


210

In [12]:
for seg_uuid in segmentation_uuids:
    segments = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
    segments = segments.json()
    sentence_list = [utt["text"] for utt in segments["utterance_set"]]
    scores = claimspotter.batch_sentence_query(sentence_list)

    for i, segment in enumerate(segments["utterance_set"]):
        requests.post(f"http://localhost:8008/api/classifications/{segment['uuid']}/", json={
            "utterance": segment["uuid"],
            "qualifier": "Checkworthiness",
            "category": "Checkworthy",
            "label": str(scores[i][1]),
            "agent": "ClaimBuster-BB"
        })

Token indices sequence length is longer than the specified maximum sequence length for this model (562 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
## JSON example request to update the database
# {
#  "utterance": "522b7392-d4bd-11ed-9fc0-00155da8ce17",
#  "qualifier": "Checkworthiness",
#  "category": "Checkworthy",
#  "label": "Quotation",
#  "agent": "test_user"
#}
# http://localhost:8000/api/classifications/<seg:uuid>/